# Dual Clustering Analysis: Comparing Numerical and Textual Clustering of Meaningful Work

This notebook conducts **two parallel clustering analyses** and compares their results:

1. **Numerical Clustering** — Cluster participants by their psychometric scale scores using direct standardization + KMeans (no PCA)
   - **Clustering A (Work-Meaning):** CMWS (28) + WAMI (10) = 38 items → primary comparison with text clusters
   - **Clustering B (Personality):** NEO (10 items) → secondary sanity-check comparison
2. **Sentence Clustering** — Cluster participants by their open-ended narrative responses using sentence embeddings + UMAP + HDBSCAN  
Three pairwise cross-clustering comparisons reveal convergence (or independence) across measurement modalities.

```
Part I    → Setup & Data
Part II   → Numerical Clustering (Work-Meaning + Personality → KMeans)
Part III  → Sentence Clustering (Embedding → UMAP → HDBSCAN)
Part IV   → Cross-Clustering Comparison (Quantitative + Qualitative)
Part V    → Summary & Methodology
```

### Research Questions
1. **RQ1 (Primary):** Do participants who cluster similarly on work-meaning survey items (CMWS + WAMI) also produce semantically similar narratives about meaningful work?
2. **RQ2 (Secondary):** Does personality-based clustering (NEO) show any association with narrative clustering or with work-meaning clustering?

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA            # used for 2D visualization only
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_extraction.text import CountVectorizer

# Clustering & Embedding
from hdbscan import HDBSCAN
from umap import UMAP
from sentence_transformers import SentenceTransformer

# cTF-IDF for text cluster interpretation
from bertopic.vectorizers import ClassTfidfTransformer

# Statistics
from scipy.stats import chi2_contingency

# Jupyter setup
%matplotlib inline
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 150

print("All packages loaded successfully.")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

TEXT_COLUMN = "Overall_Why"
RANDOM_STATE = 42

# --- Embedding ---
SELECTED_EMBEDDING = 'all-mpnet-base-v2'   # 768D, 110M params

# --- UMAP (for sentence clustering) ---
UMAP_N_NEIGHBORS = 15
UMAP_METRIC = 'cosine'

# --- HDBSCAN ---
HDBSCAN_MIN_CLUSTER = 5  

# --- Numerical Feature Groups ---
CMWS_ITEMS = [
    'CMWS_BELONGING', 'CMWS_TALK_OPEN', 'CMWS_WE_TALK', 'CMWS_WE_SUPPORT',
    'CMWS_REASSURE', 'CMWS_ENJOY',
    'CMWS_HELP_CUSTOMERS', 'CMWS_CONTRIBUTE', 'CMWS_WORTHWHILE', 'CMWS_IMPORTANT',
    'CMWS_NEW_IDEAS', 'CMWS_MATTERS', 'CMWS_ACH', 'CMWS_EXCITED',
    'CMWS_RIGHTWRONG_R', 'CMWS_DONTLIKE_R', 'CMWS_DIVORCED_R',
    'CMWS_REALITY', 'CMWS_TOLERENT', 'CMWS_MESSY',
    'CMWS_INSPIRED', 'CMWS_HOPEFUL', 'CMWS_WORK_INSPIRES', 'CMWS_SPIRITUAL',
    'CMWS_SPACE_THINK', 'CMWS_BALANCE', 'CMWS_SPACE_ME', 'CMWS_BALANCE_OTHERS',
]

WAMI_ITEMS = [
    'WAMI_Meaningful_Career', 'WAMI_Growth', 'WAMI_NO_DIFF_R',
    'WAMI_LIFES_MEANING', 'WAMI_KNOW_WHY', 'WAMI_POS_DIFFERENCE',
    'WAMI_SELF_KNOWLEDGE', 'WAMI_PURPOSE', 'WAMI_SENSEMAKE',
    'WAMI_GREATER_PURPOSE',
]

NEO_ITEMS = [
    'NEO_RESERVED', 'NEO_TRUSTING', 'NEO_LAZY', 'NEO_HANDLES_STRESS',
    'NEO_FEW_ART', 'NEO_OUTGOING', 'NEO_FAULT_OTHERS', 'NEO_THUROUGH',
    'NEO_NERVOUS', 'NEO_IMAGINATION',
]

# Two clustering groups (no PCA, standardize directly)
MEANING_CONSTRUCTS = CMWS_ITEMS + WAMI_ITEMS          # 38 items (primary)
PERSONALITY_CONSTRUCTS = NEO_ITEMS                      # 10 items (secondary)

# AC and PVC items are in the dataset but NOT used for clustering
AC_ITEMS = ['AC_1', 'AC_2', 'AC_3R', 'AC_4R', 'AC_5', 'AC_6R']
PVC_ITEMS = [
    'PVC_Power', 'PVC_Ach', 'PVC_Hedonism', 'PVC_Stimulation', 'PVC_SelfDir',
    'PVC_Univ', 'PVC_Benevolence', 'PVC_Tradition', 'PVC_Conformity', 'PVC_Security',
]
ALL_NUMERICAL = CMWS_ITEMS + WAMI_ITEMS + AC_ITEMS + PVC_ITEMS + NEO_ITEMS

print(f"Clustering groups:")
print(f"  Clustering A (Work-Meaning): {len(MEANING_CONSTRUCTS)} items (CMWS={len(CMWS_ITEMS)} + WAMI={len(WAMI_ITEMS)})")
print(f"  Clustering B (Personality):  {len(PERSONALITY_CONSTRUCTS)} items (NEO only)")
print(f"\nAdditional scales (not used for clustering):")
print(f"  AC:  {len(AC_ITEMS)} items")
print(f"  PVC: {len(PVC_ITEMS)} items")
print(f"  Total available: {len(ALL_NUMERICAL)} numerical features")

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("../data/work-stories-corpus-clean.csv")
print(f"Raw dataset: {df.shape[0]} rows, {df.shape[1]} columns")

# Check missing values in numerical features
missing = df[ALL_NUMERICAL].isnull().sum()
cols_with_missing = missing[missing > 0]
if len(cols_with_missing) > 0:
    print(f"\nColumns with missing values: {len(cols_with_missing)} columns")
    print(f"  All have {cols_with_missing.iloc[0]} missing ({cols_with_missing.iloc[0]/len(df)*100:.1f}%)")

# Drop rows with any missing numerical features or text
df_clean = df.dropna(subset=ALL_NUMERICAL + [TEXT_COLUMN]).copy()
print(f"\nAfter dropping missing values: {len(df_clean)} rows")

# Extract text documents
documents = df_clean[TEXT_COLUMN].tolist()
print(f"Documents for text clustering: {len(documents)}")

# Descriptive stats for clustering groups
print(f"\nDescriptive Statistics by Clustering Group:")
for name, items in [('CMWS (28)', CMWS_ITEMS), ('WAMI (10)', WAMI_ITEMS),
                     ('NEO (10)', NEO_ITEMS)]:
    vals = df_clean[items].values.flatten()
    print(f"  {name:12s}: mean={np.nanmean(vals):.2f}, std={np.nanstd(vals):.2f}, "
          f"range=[{np.nanmin(vals):.0f}, {np.nanmax(vals):.0f}]")

### Data Dictionary

| Scale | Items | Range | Used For | Description |
|-------|-------|-------|----------|-------------|
| CMWS | 28 | 1–5   | **Clustering A** | Comprehensive Meaningful Work Scale (Lips-Wiersma & Wright, 2012). 7 dimensions: Unity with Others, Serving Others, Expressing Full Potential, Developing Inner Self, Reality, Inspiration, Balancing Tensions |
| WAMI | 10 | 1–10  | **Clustering A** | Work and Meaning Inventory (Steger, Dik, & Duffy, 2012). 3 facets: Positive Meaning, Meaning Making, Greater Good |
| NEO | 10 | 1–6   | **Clustering B** | Big Five personality traits (NEO-FFI short form): Reserved, Trusting, Lazy, Handles Stress, Few Artistic Interests, Outgoing, Fault Others, Thorough, Nervous, Imagination |
| AC | 6 | 1–5   | *Not clustered* | Affective Commitment scale |
| PVC | 10 | 1–10  | *Not clustered* | Portrait Values Questionnaire (Schwartz, 2003). 10 value types |
| Overall_Why | free text | —     | **Text Clustering** | Open-ended narrative: "Why do you find your work meaningful (or not)?" |

**Two numerical clustering groups:**
- **Clustering A (Work-Meaning):** 38 items (CMWS + WAMI) → primary comparison with text clusters
- **Clustering B (Personality):** 10 items (NEO only) → secondary sanity check

## Part II: Numerical Clustering

Two separate clusterings on **standardized features** (no PCA):
- **Clustering A (Work-Meaning):** 38 items from CMWS + WAMI → primary analysis
- **Clustering B (Personality):** 10 NEO items → secondary sanity check

KMeans with k selected via **silhouette + elbow analysis**. We finalized cluster selection also based on our interpretation.

In [ ]:
# ============================================================
# CLUSTERING A: WORK-MEANING (CMWS + WAMI = 38 items)
# Elbow + Silhouette Analysis
# ============================================================

scaler_A = StandardScaler()
X_A = scaler_A.fit_transform(df_clean[MEANING_CONSTRUCTS])
print(f"Work-Meaning features: {X_A.shape} (n_participants x n_features)")

k_range_A = range(2, 11)
inertias_A, silhouettes_A = [], []

for k in k_range_A:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_A)
    inertias_A.append(km.inertia_)
    silhouettes_A.append(silhouette_score(X_A, labels))

# Plot elbow + silhouette (2-panel)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(list(k_range_A), inertias_A, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia (within-cluster sum of squares)')
ax.set_title('Clustering A (Work-Meaning): Elbow Method')
ax.set_xticks(list(k_range_A))

ax = axes[1]
ax.plot(list(k_range_A), silhouettes_A, 's-', color='coral', linewidth=2)
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Clustering A (Work-Meaning): Silhouette Analysis')
ax.set_xticks(list(k_range_A))

best_k_A = list(k_range_A)[np.argmax(silhouettes_A)]
ax.axvline(x=best_k_A, color='red', linestyle='--', alpha=0.7, label=f'Best k={best_k_A}')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nSilhouette scores (Work-Meaning):")
for k, sil in zip(k_range_A, silhouettes_A):
    marker = " <<<" if k == best_k_A else ""
    print(f"  k={k}: {sil:.4f}{marker}")

print(f"\nBest k by silhouette: {best_k_A} (score={max(silhouettes_A):.4f})")



In [ ]:
# ============================================================
# CLUSTERING A: FIT KMEANS + VISUALIZE

CHOSEN_K_A=2
kmeans_A = KMeans(n_clusters=CHOSEN_K_A, random_state=RANDOM_STATE, n_init=10)
labels_A = kmeans_A.fit_predict(X_A)
df_clean['num_cluster_meaning'] = labels_A

print(f"Work-Meaning KMeans: k={CHOSEN_K_A}, "
      f"silhouette={silhouette_score(X_A, labels_A):.4f}")
print(f"\nCluster sizes:")
for c in range(CHOSEN_K_A):
    n = (labels_A == c).sum()
    print(f"  Cluster {c}: {n} participants ({n/len(labels_A)*100:.1f}%)")

# 2D PCA projection FOR VISUALIZATION ONLY (not used for clustering)
pca_viz_A = PCA(n_components=2, random_state=RANDOM_STATE)
X_A_2d = pca_viz_A.fit_transform(X_A)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: 2D scatter
ax = axes[0]
colors_A = plt.cm.tab10(np.linspace(0, 1, CHOSEN_K_A))
for c in range(CHOSEN_K_A):
    mask = labels_A == c
    ax.scatter(X_A_2d[mask, 0], X_A_2d[mask, 1], c=[colors_A[c]], s=25, alpha=0.7,
               label=f'C{c} (n={mask.sum()})')
ax.set_xlabel(f'PC1 ({pca_viz_A.explained_variance_ratio_[0]*100:.1f}% var) [viz only]')
ax.set_ylabel(f'PC2 ({pca_viz_A.explained_variance_ratio_[1]*100:.1f}% var) [viz only]')
ax.set_title(f'Clustering A: Work-Meaning (k={CHOSEN_K_A})\n2D PCA projection (visualization only)')
ax.legend(fontsize=7, loc='best', ncol=2)

# Right: Cluster size bar chart
ax = axes[1]
cluster_sizes_A = [np.sum(labels_A == c) for c in range(CHOSEN_K_A)]
ax.bar(range(CHOSEN_K_A), cluster_sizes_A, color=[colors_A[c] for c in range(CHOSEN_K_A)],
       edgecolor='black', linewidth=0.5)
ax.set_xlabel('Cluster')
ax.set_ylabel('Number of Participants')
ax.set_title('Clustering A: Cluster Sizes')
ax.set_xticks(range(CHOSEN_K_A))
for c, size in enumerate(cluster_sizes_A):
    ax.text(c, size + 1, str(size), ha='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CLUSTERING A: HEATMAP + INTERPRETATION
# ============================================================

# Heatmap: mean z-scores by cluster for CMWS sub-groups and WAMI
# Group CMWS into its 7 dimensions and WAMI into 3 dimensions for readability
cmws_dims = {
    'Unity': ['CMWS_BELONGING', 'CMWS_TALK_OPEN', 'CMWS_WE_TALK',
              'CMWS_WE_SUPPORT', 'CMWS_REASSURE', 'CMWS_ENJOY'],
    'Serving': ['CMWS_HELP_CUSTOMERS', 'CMWS_CONTRIBUTE',
                'CMWS_WORTHWHILE', 'CMWS_IMPORTANT'],
    'Potential': ['CMWS_NEW_IDEAS', 'CMWS_MATTERS', 'CMWS_ACH', 'CMWS_EXCITED'],
    'InnerSelf': ['CMWS_RIGHTWRONG_R', 'CMWS_DONTLIKE_R', 'CMWS_DIVORCED_R'],
    'Reality': ['CMWS_REALITY', 'CMWS_TOLERENT', 'CMWS_MESSY'],
    'Inspiration': ['CMWS_INSPIRED', 'CMWS_HOPEFUL', 'CMWS_WORK_INSPIRES', 'CMWS_SPIRITUAL'],
    'Balance': ['CMWS_SPACE_THINK', 'CMWS_BALANCE', 'CMWS_SPACE_ME', 'CMWS_BALANCE_OTHERS'],
}
wami_dims = {
    'Positive Meaning': [
        'WAMI_Meaningful_Career', 'WAMI_KNOW_WHY', 
        'WAMI_PURPOSE', 'WAMI_SENSEMAKE'
    ],
    'Meaning Making': [
        'WAMI_Growth', 'WAMI_SELF_KNOWLEDGE', 'WAMI_GREATER_PURPOSE'
    ],
    'Greater Good': [
        'WAMI_NO_DIFF_R', 'WAMI_POS_DIFFERENCE', 'WAMI_LIFES_MEANING'
    ]
}

dim_names = list(cmws_dims.keys()) + list(wami_dims.keys())
dim_items = list(cmws_dims.values()) + list(wami_dims.values())

profile_data_A = np.zeros((CHOSEN_K_A, len(dim_names)))
for g, (name, items) in enumerate(zip(dim_names, dim_items)):
    col_indices = [MEANING_CONSTRUCTS.index(item) for item in items]
    for c in range(CHOSEN_K_A):
        mask = labels_A == c
        profile_data_A[c, g] = X_A[mask][:, col_indices].mean()

fig, ax = plt.subplots(figsize=(12, max(4, CHOSEN_K_A * 0.6)))
sns.heatmap(profile_data_A, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=dim_names,
            yticklabels=[f'C{c} (n={np.sum(labels_A==c)})' for c in range(CHOSEN_K_A)],
            ax=ax, linewidths=0.5, linecolor='white')
ax.set_title('Clustering A (Work-Meaning): Mean Z-Score by Scale Dimension')
plt.tight_layout()
plt.show()

# Top distinguishing features per cluster
print("\nCLUSTERING A: WORK-MEANING CLUSTER PROFILES")
print("=" * 70)
for c in range(CHOSEN_K_A):
    mask = labels_A == c
    n = mask.sum()
    print(f"\nCluster {c} (n={n}, {n/len(labels_A)*100:.1f}%)")
    print("-" * 50)
    
    # Mean raw scores by scale
    cmws_mean = df_clean.loc[mask, CMWS_ITEMS].values.mean()
    wami_mean = df_clean.loc[mask, WAMI_ITEMS].values.mean()
    print(f"  CMWS mean: {cmws_mean:.2f}  |  WAMI mean: {wami_mean:.2f}")
    
    # Top 5 distinguishing features
    cluster_means = X_A[mask].mean(axis=0)
    top_features = np.argsort(-np.abs(cluster_means))[:5]
    print(f"  Top distinguishing features:")
    for feat_idx in top_features:
        feat_name = MEANING_CONSTRUCTS[feat_idx]
        z = cluster_means[feat_idx]
        direction = "HIGH" if z > 0 else "LOW"
        print(f"    {feat_name}: z={z:+.2f} ({direction})")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Setup the x-axis labels from your dim_names
# dim_names = ['Unity', 'Serving', 'Potential', 'InnerSelf', 'Reality', 
#              'Inspiration', 'Balance', 'Positive Meaning', 'Meaning Making', 'Greater Good']
x_indices = np.arange(len(dim_names))

plt.figure(figsize=(14, 7))

# Define distinct colors and markers for the clusters
# Using high-contrast colors for a professional look
line_colors = ['#1f77b4', '#ff7f0e', '#2ca02c'] 
markers = ['o', 'D', '^']

for i in range(CHOSEN_K_A):
    plt.plot(x_indices, profile_data_A[i], 
             label=f'Cluster {i} (n={np.sum(labels_A==i)})',
             linestyle='-',
             marker=markers[i % len(markers)],
             color=line_colors[i % len(line_colors)],
             linewidth=3, 
             markersize=10,
             markeredgecolor='white',
             alpha=0.9)

# 2. Add professional formatting
plt.xticks(x_indices, dim_names, rotation=35, ha='right', fontsize=10)
plt.axhline(0, color='black', linestyle='-', alpha=0.3, linewidth=1.5) # Mean Z-score baseline
plt.ylabel('Mean Z-Score', fontsize=12, fontweight='bold')
plt.xlabel('CMWS & WAMI Dimensions', fontsize=12, fontweight='bold')
plt.title('Work-Meaning Profile Comparison across Clusters', fontsize=14, fontweight='bold', pad=20)

# Standardize Y-axis for Z-scores
plt.ylim(-1.5, 1.5)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Add a visual separator between CMWS and WAMI for clarity
plt.axvline(6.5, color='gray', linestyle=':', alpha=0.5)
plt.text(3, 1.3, 'CMWS Dimensions', horizontalalignment='center', color='gray', fontweight='bold')
plt.text(8, 1.3, 'WAMI Facets', horizontalalignment='center', color='gray', fontweight='bold')

plt.legend(frameon=True, loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

### Clustering B: Personality Constructs (NEO, 10 items)

Secondary clustering as a **sanity check** and pattern discovery step. Only NEO Big Five items are used (10 items).

In [ ]:
# ============================================================
# CLUSTERING B: PERSONALITY (NEO = 10 items)
# Elbow + Silhouette Analysis
# ============================================================

scaler_B = StandardScaler()
X_B = scaler_B.fit_transform(df_clean[PERSONALITY_CONSTRUCTS])
print(f"Personality features: {X_B.shape} (n_participants x n_features)")

k_range_B = range(2, 11)
inertias_B, silhouettes_B = [], []

for k in k_range_B:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_B)
    inertias_B.append(km.inertia_)
    silhouettes_B.append(silhouette_score(X_B, labels))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(list(k_range_B), inertias_B, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia')
ax.set_title('Clustering B (Personality/NEO): Elbow Method')
ax.set_xticks(list(k_range_B))

ax = axes[1]
ax.plot(list(k_range_B), silhouettes_B, 's-', color='coral', linewidth=2)
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Clustering B (Personality/NEO): Silhouette Analysis')
ax.set_xticks(list(k_range_B))

best_k_B = list(k_range_B)[np.argmax(silhouettes_B)]
ax.axvline(x=best_k_B, color='red', linestyle='--', alpha=0.7, label=f'Best k={best_k_B}')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nSilhouette scores (Personality/NEO):")
for k, sil in zip(k_range_B, silhouettes_B):
    marker = " <<<" if k == best_k_B else ""
    print(f"  k={k}: {sil:.4f}{marker}")



In [ ]:
# ============================================================
# CLUSTERING B: FIT KMEANS + VISUALIZE
# ============================================================
CHOSEN_K_B = 3
kmeans_B = KMeans(n_clusters=CHOSEN_K_B, random_state=RANDOM_STATE, n_init=10)
labels_B = kmeans_B.fit_predict(X_B)
df_clean['num_cluster_personality'] = labels_B

print(f"Personality KMeans: k={CHOSEN_K_B}, "
      f"silhouette={silhouette_score(X_B, labels_B):.4f}")
print(f"\nCluster sizes:")
for c in range(CHOSEN_K_B):
    n = (labels_B == c).sum()
    print(f"  Cluster {c}: {n} participants ({n/len(labels_B)*100:.1f}%)")

# 2D PCA projection for visualization only
pca_viz_B = PCA(n_components=2, random_state=RANDOM_STATE)
X_B_2d = pca_viz_B.fit_transform(X_B)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: 2D scatter
ax = axes[0]
colors_B = plt.cm.tab10(np.linspace(0, 1, CHOSEN_K_B))
for c in range(CHOSEN_K_B):
    mask = labels_B == c
    ax.scatter(X_B_2d[mask, 0], X_B_2d[mask, 1], c=[colors_B[c]], s=25, alpha=0.7,
               label=f'C{c} (n={mask.sum()})')
ax.set_xlabel(f'PC1 ({pca_viz_B.explained_variance_ratio_[0]*100:.1f}% var) [viz only]')
ax.set_ylabel(f'PC2 ({pca_viz_B.explained_variance_ratio_[1]*100:.1f}% var) [viz only]')
ax.set_title(f'Clustering B: Personality/NEO (k={CHOSEN_K_B})\n2D PCA projection (visualization only)')
ax.legend(fontsize=7, loc='best', ncol=2)

# Right: Heatmap of NEO trait profiles
ax = axes[1]
profile_data_B = np.zeros((CHOSEN_K_B, len(NEO_ITEMS)))
neo_short = [n.replace('NEO_', '') for n in NEO_ITEMS]
for g, item in enumerate(NEO_ITEMS):
    col_idx = PERSONALITY_CONSTRUCTS.index(item)
    for c in range(CHOSEN_K_B):
        mask = labels_B == c
        profile_data_B[c, g] = X_B[mask][:, col_idx].mean()

sns.heatmap(profile_data_B, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=neo_short,
            yticklabels=[f'C{c} (n={np.sum(labels_B==c)})' for c in range(CHOSEN_K_B)],
            ax=ax, linewidths=0.5, linecolor='white')
ax.set_title('Clustering B: NEO Trait Profiles (Mean Z-Score)')

plt.tight_layout()
plt.show()

In [ ]:
x_indices = np.arange(len(neo_short))

plt.figure(figsize=(12, 7))

line_colors = ['#1f77b4', '#d62728', '#2ca02c','#2ca02c'] 
styles = ['-', '-', '-', '-']  # Keeping lines solid for cleaner comparison
markers = ['o', 's', '^', 'o']

for i in range(CHOSEN_K_B):
    plt.plot(x_indices, profile_data_B[i], 
             label=f'Cluster {i} (n={np.sum(labels_B==i)})',
             linestyle=styles[i],
             marker=markers[i],
             color=line_colors[i],
             linewidth=2.5, 
             markersize=8,
             markeredgecolor='white', # High contrast markers
             alpha=0.8)

# 2. Add professional formatting
plt.xticks(x_indices, neo_short, rotation=45, ha='right')
plt.axhline(0, color='black', linestyle='-', alpha=0.3, linewidth=1) # Mean baseline
plt.ylabel('Mean Z-Score', fontsize=12, fontweight='bold')
plt.xlabel('NEO Personality Variables', fontsize=12, fontweight='bold')
plt.title('Personality Profile Comparison across Clusters', fontsize=14, fontweight='bold', pad=20)

# Setting Y-axis limits to capture the range (-1.5 to 1.5 is standard for Z-scores)
plt.ylim(-1.5, 1.5)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Placing legend outside the plot area if it gets too crowded
plt.legend(frameon=True, loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

#### Why we are choosing k=3 despite higher Silhouette score for k=2? 

Our k=3 model perfectly isolates Cluster 1 (n=72) as a "Low Conscientiousness" group with high LAZY (1.03) and low THOROUGH (−0.80) scores. At k=2, these nuances become "averaged out," our first thought was that it makes it harder to predict workplace behaviors like dependability or impulse control (we have more detail on labeling process in the report).

In [ ]:
# ============================================================
# CLUSTERING B: INTERPRETATION
# ============================================================

print("CLUSTERING B: PERSONALITY (NEO) CLUSTER PROFILES")
print("=" * 70)
for c in range(CHOSEN_K_B):
    mask = labels_B == c
    n = mask.sum()
    print(f"\nCluster {c} (n={n}, {n/len(labels_B)*100:.1f}%)")
    print("-" * 50)
    
    # Mean z-scores for each NEO trait
    cluster_means = X_B[mask].mean(axis=0)
    sorted_idx = np.argsort(-np.abs(cluster_means))
    print(f"  NEO trait profile (z-scores):")
    for idx in sorted_idx:
        trait = NEO_ITEMS[idx].replace('NEO_', '')
        z = cluster_means[idx]
        bar = '+' * int(abs(z) * 10) if z > 0 else '-' * int(abs(z) * 10)
        print(f"    {trait:15s}: z={z:+.2f} {bar}")

## Part III: Sentence Clustering

Cluster participants by the semantic content of their open-ended narratives about meaningful work.

**Pipeline:** Sentence embedding (all-mpnet-base-v2) → UMAP (dimensionality reduction) → HDBSCAN (density-based clustering)

**Key design decisions:**
- ~30% outliers is normal for open-ended responses
- Silhouette is reported for transparency could be **misleading for HDBSCAN** (density-based); interpretability and cluster persistence matter more
- Interpret clusters with **cTF-IDF** and representative documents

In [ ]:
%%time
# ============================================================
# EMBED NARRATIVES
# ============================================================

embedding_model = SentenceTransformer(SELECTED_EMBEDDING)
embeddings = embedding_model.encode(documents, show_progress_bar=True, batch_size=32)

print(f"Embedding model: {SELECTED_EMBEDDING}")
print(f"Embeddings shape: {embeddings.shape}")
print(f"  {embeddings.shape[0]} documents x {embeddings.shape[1]} dimensions")

In [ ]:
%%time
# ============================================================
# UMAP DIMENSIONALITY REDUCTION — FIND OPTIMAL n_components
# ============================================================

# Test different n_components, evaluate via silhouette on HDBSCAN output
# Note: silhouette is for transparency only; interpretability is the real criterion
n_components_to_test = [5, 10, 15, 20, 30, 50]
umap_results = []

print(f"Testing UMAP n_components with HDBSCAN (min_cluster_size={HDBSCAN_MIN_CLUSTER}):")
print("-" * 70)

for n_comp in n_components_to_test:
    umap_model = UMAP(
        n_neighbors=UMAP_N_NEIGHBORS, n_components=n_comp,
        metric=UMAP_METRIC, min_dist=0.0, random_state=RANDOM_STATE
    )
    X_reduced = umap_model.fit_transform(embeddings)
    
    clusterer = HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER,
        metric='euclidean',
        cluster_selection_method='eom'
    )
    labels = clusterer.fit_predict(X_reduced)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_outliers = (labels == -1).sum()
    outlier_pct = n_outliers / len(labels) * 100
    
    # Silhouette on non-outlier points (for transparency)
    non_outlier = labels != -1
    if non_outlier.sum() >= 2 and n_clusters >= 2:
        sil = silhouette_score(X_reduced[non_outlier], labels[non_outlier])
    else:
        sil = -1
    
    umap_results.append({
        'n_components': n_comp, 'n_clusters': n_clusters,
        'outlier_pct': outlier_pct, 'silhouette': sil
    })
    print(f"  n_comp={n_comp:3d}: {n_clusters:2d} clusters, "
          f"{outlier_pct:5.1f}% outliers, silhouette={sil:.4f}")

# Pick configuration with best silhouette (cluster-count agnostic)
valid = [r for r in umap_results if r['n_clusters'] >= 2]

if valid:
    best = max(valid, key=lambda x: x['silhouette'])
    print(f"\nSelected: n_components={best['n_components']} "
          f"({best['n_clusters']} clusters, silhouette={best['silhouette']:.4f})")
    BEST_UMAP_N = best['n_components']
else:
    print("\nNo valid configuration. Using n_components=10.")
    BEST_UMAP_N = 10

In [ ]:
# ============================================================
# HDBSCAN min_cluster_size SWEEP: Bootstrap ARI + cTF-IDF
# ============================================================
# Sweep min_cluster_size to find the most stable cluster count.
# Selection criterion: Bootstrap Adjusted Rand Index (ARI)
#   — measures reproducibility under 80% resampling (5 resamples).
# Interpretability check: cTF-IDF top 5 words per cluster and representative documents.

mcs_values = [5, 6, 7, 8, 10, 12, 15, 20]
sweep_results = []
sweep_topics = {}  # mcs -> {cluster_id: [top words]}

print("min_cluster_size Sweep on UMAP-reduced embeddings")
print(f"Embeddings shape: {X_text.shape}")
print("=" * 75)

for mcs in mcs_values:
    clusterer = HDBSCAN(min_cluster_size=mcs, metric='euclidean',
                        cluster_selection_method='eom')
    labels = clusterer.fit_predict(X_text)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_outliers = (labels == -1).sum()
    pct_outlier = n_outliers / len(labels) * 100
    sizes = [int((labels == c).sum()) for c in sorted(set(labels) - {-1})]

    # --- Bootstrap Stability (ARI, 5 resamples at 80%) ---
    n_boot = 5
    boot_aris = []
    rng = np.random.RandomState(RANDOM_STATE)
    for _ in range(n_boot):
        idx = rng.choice(len(X_text), size=int(0.8 * len(X_text)), replace=False)
        X_boot = X_text[idx]
        boot_clusterer = HDBSCAN(min_cluster_size=mcs, metric='euclidean',
                                  cluster_selection_method='eom')
        boot_labels = boot_clusterer.fit_predict(X_boot)
        full_sub = labels[idx]
        n_boot_clusters = len(set(boot_labels)) - (1 if -1 in boot_labels else 0)
        n_full_sub_clusters = len(set(full_sub)) - (1 if -1 in full_sub else 0)
        if n_boot_clusters >= 2 and n_full_sub_clusters >= 2:
            boot_aris.append(adjusted_rand_score(full_sub, boot_labels))
    boot_ari = float(np.mean(boot_aris)) if len(boot_aris) > 0 else np.nan

    # --- cTF-IDF top 5 words ---
    cids = sorted(set(labels) - {-1})
    top_words_dict = {}
    if len(cids) >= 2:
        dpc = [' '.join([documents[i] for i in range(len(documents))
                         if labels[i] == c]) for c in cids]
        # Adjust min_df for small cluster counts to avoid max_df < min_df error
        sw_min_df = 1 if len(cids) <= 3 else 2
        cv_sw = CountVectorizer(stop_words='english', min_df=sw_min_df, max_df=0.95)
        try:
            X_c = cv_sw.fit_transform(dpc)
            ctf_sw = ClassTfidfTransformer(reduce_frequent_words=True)
            X_ctf = ctf_sw.fit_transform(X_c)
            fnames = cv_sw.get_feature_names_out()
            for i, c in enumerate(cids):
                scores = X_ctf[i].toarray().flatten()
                top5 = [fnames[j] for j in scores.argsort()[-5:][::-1]]
                top_words_dict[c] = top5
        except ValueError:
            pass  # Skip cTF-IDF if vectorization fails

    sweep_topics[mcs] = top_words_dict
    sweep_results.append({
        'mcs': mcs, 'n_clusters': n_clusters,
        'outlier_pct': pct_outlier, 'sizes': sizes,
        'boot_ari': boot_ari
    })

    # Per-mcs printout
    print(f"\nmcs={mcs:3d}: {n_clusters} clusters, {pct_outlier:.1f}% outliers, Bootstrap ARI={boot_ari:.4f}")
    print(f"  Sizes: {sizes}")
    for c in cids:
        if c in top_words_dict:
            print(f"  C{c} (n={(labels==c).sum():3d}): {', '.join(top_words_dict[c])}")

# ============================================================
# SUMMARY COMPARISON TABLE
# ============================================================
print("\n" + "=" * 75)
print("SUMMARY COMPARISON TABLE")
print("=" * 75)

summary_df = pd.DataFrame(sweep_results)
summary_df = summary_df[['mcs', 'n_clusters', 'outlier_pct', 'boot_ari']]
summary_df.columns = ['mcs', 'clusters', 'outlier%', 'boot_ARI']

best_idx = summary_df['boot_ARI'].idxmax()

print(summary_df.to_string(index=False, float_format='%.4f'))
print(f"\n  Best Bootstrap ARI: mcs={summary_df.loc[best_idx, 'mcs']}  <<<")
print(f"\n  boot_ARI: Bootstrap stability (5x 80% resample); higher = more reproducible clustering")

# ============================================================
# >>> SET OUR CHOSEN min_cluster_size <<<
# ============================================================
CHOSEN_MCS = 7  # <-- THIS is based on the sweep results above

# Re-fit HDBSCAN with chosen mcs and overwrite downstream variables
HDBSCAN_MIN_CLUSTER = CHOSEN_MCS
hdbscan_final = HDBSCAN(
    min_cluster_size=CHOSEN_MCS,
    metric='euclidean',
    cluster_selection_method='eom'
)
text_cluster_labels = hdbscan_final.fit_predict(X_text)
df_clean['text_cluster'] = text_cluster_labels

# Update derived variables for downstream cells
n_text_clusters = len(set(text_cluster_labels)) - (1 if -1 in text_cluster_labels else 0)
n_text_outliers = (text_cluster_labels == -1).sum()
outlier_pct = n_text_outliers / len(text_cluster_labels) * 100
cluster_ids = sorted(set(text_cluster_labels) - {-1})

print(f"\n{'='*60}")
print(f"APPLIED: min_cluster_size = {CHOSEN_MCS}")
print(f"  Clusters: {n_text_clusters}")
print(f"  Outliers: {n_text_outliers} ({outlier_pct:.1f}%)")
for c in cluster_ids:
    n = (text_cluster_labels == c).sum()
    print(f"  Cluster {c}: {n} participants ({n/len(text_cluster_labels)*100:.1f}%)")

In [ ]:
# ============================================================
# TEXT CLUSTER VISUALIZATION
# ============================================================

# 2D UMAP for visualization
umap_2d = UMAP(n_neighbors=UMAP_N_NEIGHBORS, n_components=2,
               metric=UMAP_METRIC, min_dist=0.3, random_state=RANDOM_STATE)
text_2d = umap_2d.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: HDBSCAN scatter
ax = axes[0]
outlier_mask = text_cluster_labels == -1
if outlier_mask.any():
    ax.scatter(text_2d[outlier_mask, 0], text_2d[outlier_mask, 1],
               c='lightgray', s=15, alpha=0.4, label=f'Outliers (n={outlier_mask.sum()})')
colors_text = plt.cm.tab10(np.linspace(0, 1, max(len(cluster_ids), 1)))
for i, c in enumerate(cluster_ids):
    mask = text_cluster_labels == c
    ax.scatter(text_2d[mask, 0], text_2d[mask, 1],
               c=[colors_text[i % 10]], s=25, alpha=0.7,
               label=f'C{c} (n={mask.sum()})')
ax.set_title(f'HDBSCAN (mcs={HDBSCAN_MIN_CLUSTER})\n'
             f'{n_text_clusters} clusters, {outlier_pct:.1f}% outliers')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend(fontsize=7, loc='best', ncol=2)

# Right: Cluster size bar chart
ax = axes[1]
sizes = [(text_cluster_labels == c).sum() for c in cluster_ids]
bar_colors = [colors_text[i % 10] for i in range(len(cluster_ids))]
ax.bar(range(len(cluster_ids)), sizes, color=bar_colors,
       edgecolor='black', linewidth=0.5)
ax.set_xlabel('Cluster')
ax.set_ylabel('Number of Participants')
ax.set_title('Text Cluster Sizes (Excluding Outliers)')
ax.set_xticks(range(len(cluster_ids)))
ax.set_xticklabels([f'C{c}' for c in cluster_ids])
for i, (c, size) in enumerate(zip(cluster_ids, sizes)):
    ax.text(i, size + 0.5, str(size), ha='center', fontsize=8)

plt.suptitle('Sentence Clustering: HDBSCAN Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TEXT CLUSTER INTERPRETATION: cTF-IDF
# ============================================================

# Concatenate all documents per cluster into a single pseudo-document
cluster_ids = sorted(set(text_cluster_labels) - {-1})
docs_per_class = []
for c in cluster_ids:
    cluster_docs = [documents[i] for i in range(len(documents)) if text_cluster_labels[i] == c]
    docs_per_class.append(' '.join(cluster_docs))

# Fit CountVectorizer + ClassTfidfTransformer
cv = CountVectorizer(ngram_range=(1,2), stop_words='english', min_df=2, max_df=0.95)
X_counts = cv.fit_transform(docs_per_class)
ctfidf = ClassTfidfTransformer(reduce_frequent_words=True)
X_ctfidf = ctfidf.fit_transform(X_counts)
feature_names = cv.get_feature_names_out()

print("TEXT CLUSTER INTERPRETATION (cTF-IDF)")
print("=" * 70)
print(f"Vocabulary size: {len(feature_names)} terms")
print(f"Note: cTF-IDF highlights words DISTINCTIVE to each cluster,")
print(f"      suppressing ubiquitous words like 'meaningful'.")

for i, c in enumerate(cluster_ids):
    mask = text_cluster_labels == c
    n = mask.sum()
    scores = X_ctfidf[i].toarray().flatten()
    top_idx = scores.argsort()[-10:][::-1]
    top_words = [(feature_names[j], scores[j]) for j in top_idx]
    
    print(f"\nCluster {c} (n={n})")
    print(f"  Top 10 distinctive words (cTF-IDF):")
    for word, score in top_words:
        print(f"    {word:20s} {score:.4f}")
    
    # Representative documents (closest to embedding centroid)
    cluster_embeddings = embeddings[mask]
    centroid = cluster_embeddings.mean(axis=0)
    distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)
    cluster_docs_list = [documents[j] for j in range(len(documents)) if text_cluster_labels[j] == c]
    closest_idx = np.argsort(distances)[:3]
    
    print(f"  Representative documents:")
    for rank, idx in enumerate(closest_idx):
        doc_text = cluster_docs_list[idx]#[:200]
        print(f"    [{rank+1}] \"{doc_text}...\"")

## Part IV: Cross-Clustering Comparison

**Approach - Quantitative:** Chi-squared test for:
1. Work-Meaning clusters × Personality clusters (numerical-to-numerical)
2. Work-Meaning clusters × Text clusters (primary vs. text)
3. Personality clusters × Text clusters (secondary plug-in)


In [ ]:
# ============================================================
# APPROACH - QUANTITATIVE (Chi-squared test)
# ============================================================

results = {}

# ----------------------------------------------------------
# COMPARISON 1: Work-Meaning x Personality (all participants)
# No outlier filtering — both are KMeans, all 194 assigned
# ----------------------------------------------------------
print("=" * 70)
print("1. Work-Meaning Clusters x Personality Clusters")
print(f"   (All {len(df_clean)} participants — no outlier exclusion)")
print("=" * 70)

ct_wp = pd.crosstab(
    df_clean['num_cluster_meaning'],
    df_clean['num_cluster_personality'],
    margins=True
)
print(f"\nContingency Table:")
print(ct_wp)

ct_wp_vals = ct_wp.iloc[:-1, :-1]
chi2_wp, p_wp, dof_wp, expected_wp = chi2_contingency(ct_wp_vals)
low_expected_wp = (expected_wp < 5).sum() / expected_wp.size * 100

print(f"\nStatistics:")
print(f"  Chi-squared: {chi2_wp:.4f}")
print(f"  p-value: {p_wp:.4f}")
print(f"  df: {dof_wp}")
print(f"  {'Significant' if p_wp < 0.05 else 'Not significant'} (alpha=0.05)")

if low_expected_wp > 20:
    print(f"  WARNING: {low_expected_wp:.0f}% of cells have expected count < 5.")
    print(f"  Chi-squared results should be interpreted with caution.")

results['Work-Meaning x Personality'] = {
    'chi2': chi2_wp, 'p': p_wp, 'dof': dof_wp,
    'low_expected_pct': low_expected_wp, 'ct': ct_wp_vals, 'expected': expected_wp
}

# ----------------------------------------------------------
# COMPARISONS 2-3: Numerical x Text clusters (outlier-filtered)
# ----------------------------------------------------------
non_outlier = text_cluster_labels != -1
n_non_outlier = non_outlier.sum()
print(f"\n\nNon-outlier participants for text comparisons: {n_non_outlier} / {len(text_cluster_labels)}")
print(f"(Outliers excluded: {(~non_outlier).sum()}, {(~non_outlier).sum()/len(text_cluster_labels)*100:.1f}%)")

for idx, (name, col) in enumerate([('Work-Meaning', 'num_cluster_meaning'),
                                     ('Personality (NEO)', 'num_cluster_personality')], start=2):
    print(f"\n{'='*70}")
    print(f"{idx}. {name} Clusters x Text Clusters")
    print(f"{'='*70}")
    
    # Contingency table
    ct = pd.crosstab(
        df_clean.loc[non_outlier, col],
        df_clean.loc[non_outlier, 'text_cluster'],
        margins=True
    )
    print(f"\nContingency Table:")
    print(ct)
    
    # Chi-squared
    ct_vals = ct.iloc[:-1, :-1]
    chi2, p, dof, expected = chi2_contingency(ct_vals)
    
    # Check expected count validity
    low_expected = (expected < 5).sum() / expected.size * 100
    
    print(f"\nStatistics:")
    print(f"  Chi-squared: {chi2:.4f}")
    print(f"  p-value: {p:.4f}")
    print(f"  df: {dof}")
    print(f"  {'Significant' if p < 0.05 else 'Not significant'} (alpha=0.05)")
    
    if low_expected > 20:
        print(f"  WARNING: {low_expected:.0f}% of cells have expected count < 5.")
        print(f"  Chi-squared results should be interpreted with caution.")
    
    results[name] = {
        'chi2': chi2, 'p': p, 'dof': dof,
        'low_expected_pct': low_expected, 'ct': ct_vals, 'expected': expected
    }

In [ ]:
# ============================================================
# VISUALIZATION (3-panel heatmaps)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# Panel 1: Work-Meaning x Personality
ax = axes[0]
ct_wp_plot = results['Work-Meaning x Personality']['ct']
sns.heatmap(ct_wp_plot, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='white')
r_wp = results['Work-Meaning x Personality']
ax.set_title(f'Work-Meaning x Personality\n'
             f'\u03c7\u00b2={r_wp["chi2"]:.2f}, p={r_wp["p"]:.4f}')
ax.set_xlabel('Personality Cluster')
ax.set_ylabel('Work-Meaning Cluster')

# Panel 2: Work-Meaning x Text
ax = axes[1]
ct_wm_plot = results['Work-Meaning']['ct']
sns.heatmap(ct_wm_plot, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='white')
r_wm = results['Work-Meaning']
ax.set_title(f'Work-Meaning x Text\n'
             f'\u03c7\u00b2={r_wm["chi2"]:.2f}, p={r_wm["p"]:.4f}')
ax.set_xlabel('Text Cluster')
ax.set_ylabel('Work-Meaning Cluster')

# Panel 3: Personality x Text
ax = axes[2]
ct_pt_plot = results['Personality (NEO)']['ct']
sns.heatmap(ct_pt_plot, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='white')
r_pt = results['Personality (NEO)']
ax.set_title(f'Personality x Text\n'
             f'\u03c7\u00b2={r_pt["chi2"]:.2f}, p={r_pt["p"]:.4f}')
ax.set_xlabel('Text Cluster')
ax.set_ylabel('Personality Cluster')

plt.suptitle('Quantitative Comparison: All Three Cross-Clustering Pairs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Cross-Comparison Summary

**Quantitative findings:**
- Work-Meaning × Personality: Reveals whether work-meaning profiles (CMWS + WAMI) are associated with personality profiles (NEO). This comparison uses all participants (no outlier exclusion).
- Work-Meaning × Text: Chi-squared test indicates whether survey-based clusters and narrative-based clusters are statistically independent.
- Personality × Text: The secondary comparison reveals whether Big Five personality traits clusters like narrative style.


## Part V: Summary & Methodology

In [ ]:
# ============================================================
# SUMMARY
# ============================================================

print("=" * 75)
print("ANALYSIS SUMMARY")
print("=" * 75)

# Numerical clustering A
print(f"\n--- Clustering A: Work-Meaning ---")
print(f"  Features: {len(MEANING_CONSTRUCTS)} items (CMWS={len(CMWS_ITEMS)} + WAMI={len(WAMI_ITEMS)})")
print(f"  Method: Standardize (z-score) -> KMeans (no PCA)")
print(f"  k = {CHOSEN_K_A}, silhouette = {silhouette_score(X_A, labels_A):.4f}")
for c in range(CHOSEN_K_A):
    print(f"    Cluster {c}: n={(labels_A == c).sum()}")

# Numerical clustering B
print(f"\n--- Clustering B: Personality (NEO) ---")
print(f"  Features: {len(PERSONALITY_CONSTRUCTS)} items (NEO only)")
print(f"  Method: Standardize (z-score) -> KMeans (no PCA)")
print(f"  k = {CHOSEN_K_B}, silhouette = {silhouette_score(X_B, labels_B):.4f}")
for c in range(CHOSEN_K_B):
    print(f"    Cluster {c}: n={(labels_B == c).sum()}")

# Text clustering
print(f"\n--- Text Clustering ---")
print(f"  Embedding: {SELECTED_EMBEDDING} ({embeddings.shape[1]}D)")
print(f"  UMAP: n_components={BEST_UMAP_N}")
print(f"  HDBSCAN: min_cluster_size={HDBSCAN_MIN_CLUSTER}")
print(f"  Clusters: {n_text_clusters}, Outliers: {n_text_outliers} ({outlier_pct:.1f}%)")
print(f"  Interpretation: cTF-IDF (reduce_frequent_words=True)")

# Cross-comparison
print(f"\n--- Cross-Clustering Comparison (Chi-squared) ---")
for name in ['Work-Meaning x Personality', 'Work-Meaning', 'Personality (NEO)']:
    r = results[name]
    sig = 'p<0.05' if r['p'] < 0.05 else f'p={r["p"]:.3f}'
    label = f"{name} x Text" if name in ['Work-Meaning', 'Personality (NEO)'] else name
    n_used = len(df_clean) if name == 'Work-Meaning x Personality' else non_outlier.sum()
    print(f"  {label} (n={n_used}):")
    print(f"    Chi-squared={r['chi2']:.4f}, {sig}")
    if r['low_expected_pct'] > 20:
        print(f"    (Caution: {r['low_expected_pct']:.0f}% expected counts < 5)")